# DSC670 Final Project - NoteMorphAI (Notely)

**Student:** Komal Shahid  
**Course:** DSC670 - Machine Learning Applications  
**Semester:** Fall 2024  
**Institution:** Bellevue University  
**Project:** AI-Powered Note Transformation System

---

## Project Overview

NoteMorphAI (Notely) is an innovative AI-powered note transformation system that converts user input into beautifully designed, interactive note templates. The system features multiple template styles designed for various functional use cases and purposes.

### Template Categories by Use Case:
- 📝 **Meeting Notes** - Structured templates for capturing meeting discussions and action items
- 📋 **Project Planning** - Comprehensive templates for project coordination and planning
- 📰 **Blog/Article Templates** - Professional layouts for content creation and publishing
- 🎓 **Academic Notes (Cornell Style)** - Structured educational templates for learning and study
- 📊 **Infographic Templates** - Visual templates for data presentation and education
- 📚 **Step-by-Step Guides** - Sequential instruction templates for tutorials and processes

### Template Styling Themes:
- **Professional** - Clean, business-oriented layouts
- **Educational** - Academic-focused with clear structure
- **Visual/Creative** - Image-rich and engaging designs
- **Minimalist** - Simple, distraction-free layouts
- **Interactive** - Web-based templates with dynamic elements

### System Architecture

The system consists of several key components:
1. **Template Generator** - Core template creation engine
2. **Template Manager** - Template storage and organization
3. **Showcase Templates** - Pre-built template library
4. **Creative Templates** - Advanced template variations
5. **Streamlit App** - Web interface for user interaction
6. **HTML Template Data** - Structured template content in JSON format

### Sample Content Examples:
The system demonstrates its versatility using various content examples including climate change education, nutrition guides, and project planning scenarios to showcase how the same template frameworks can be adapted for different subject matters.

## Import Statements

Essential libraries for the note transformation system:

In [1]:
# Core Python libraries
import os
import sys
import json
import shutil
import webbrowser
import tempfile
import datetime
import random
from pathlib import Path
import base64
import time

# Data processing
import numpy as np

# Image processing library
try:
    from PIL import Image, ImageDraw, ImageFont
    print("✅ PIL/Pillow available for image processing")
except ImportError:
    print("❌ PIL/Pillow not available - install with: pip install Pillow")

print("✅ All dependencies imported successfully")

✅ PIL/Pillow available for image processing
✅ All dependencies imported successfully


## Project Setup

Configure project paths and directories:

In [2]:
# Set up project paths
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "milestone3":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent  # Navigate to notely directory
    
SRC_PATH = PROJECT_ROOT / "src"
DATA_PATH = PROJECT_ROOT / "data"
OUTPUT_PATH = PROJECT_ROOT / "output"
TEMPLATES_PATH = OUTPUT_PATH / "templates"

# Create directories if they don't exist
OUTPUT_PATH.mkdir(exist_ok=True)
TEMPLATES_PATH.mkdir(exist_ok=True)

print(f"📁 Project root: {PROJECT_ROOT}")
print(f"📁 Source code: {SRC_PATH}")
print(f"📁 Data directory: {DATA_PATH}")
print(f"📁 Templates output: {TEMPLATES_PATH}")
print("✅ Project setup complete")

📁 Project root: /Users/komalshahid/Desktop/Bellevue University/DSC670/term_project/notely
📁 Source code: /Users/komalshahid/Desktop/Bellevue University/DSC670/term_project/notely/src
📁 Data directory: /Users/komalshahid/Desktop/Bellevue University/DSC670/term_project/notely/data
📁 Templates output: /Users/komalshahid/Desktop/Bellevue University/DSC670/term_project/notely/output/templates
✅ Project setup complete


## Template Data Loading

Load HTML template structures from JSON file for clean code organization:

In [3]:
def load_template_data():
    """Load HTML template data from JSON file"""
    template_file = DATA_PATH / "html_templates.json"
    
    try:
        with open(template_file, "r", encoding="utf-8") as f:
            template_data = json.load(f)
        print(f"✅ Template data loaded from {template_file}")
        return template_data
    except FileNotFoundError:
        print(f"❌ Template file not found: {template_file}")
        return None
    except json.JSONDecodeError as e:
        print(f"❌ Error parsing template JSON: {e}")
        return None

# Load template data
TEMPLATE_DATA = load_template_data()

if TEMPLATE_DATA:
    print(f"📄 Template definitions: {len(TEMPLATE_DATA['templates'])}")
    print(f"🎨 Style variants: {len(TEMPLATE_DATA['common_styles'])}")
else:
    print("⚠️ Using fallback template content")

✅ Template data loaded from /Users/komalshahid/Desktop/Bellevue University/DSC670/term_project/notely/data/html_templates.json
📄 Template definitions: 4
🎨 Style variants: 4


## Template Manager Class

Manages template storage, organization, and metadata tracking:

In [4]:
class TemplateManager:
    """
    Manages templates for the Notely application.
    
    Features:
    - Template storage and organization
    - Template metadata tracking
    - Template search and retrieval
    - Template collections
    """
    
    def __init__(self, template_dir="templates"):
        """Initialize the template manager with a template directory"""
        self.template_dir = Path(template_dir)
        self.template_dir.mkdir(exist_ok=True)
        
        # Create metadata file if it doesn't exist
        self.metadata_file = self.template_dir / "template_metadata.json"
        if not self.metadata_file.exists():
            self._initialize_metadata()
    
    def _initialize_metadata(self):
        """Initialize an empty metadata file"""
        metadata = {
            "templates": {},
            "collections": {},
            "last_updated": datetime.datetime.now().isoformat()
        }
        
        with open(self.metadata_file, "w") as f:
            json.dump(metadata, f, indent=2)
    
    def _load_metadata(self):
        """Load metadata from file"""
        try:
            with open(self.metadata_file, "r") as f:
                return json.load(f)
        except (FileNotFoundError, json.JSONDecodeError):
            self._initialize_metadata()
            with open(self.metadata_file, "r") as f:
                return json.load(f)
    
    def _save_metadata(self, metadata):
        """Save metadata to file"""
        metadata["last_updated"] = datetime.datetime.now().isoformat()
        with open(self.metadata_file, "w") as f:
            json.dump(metadata, f, indent=2)
    
    def add_template(self, template_path, name=None, description=None, tags=None):
        """Add a template to the manager"""
        template_filename = os.path.basename(template_path)
        
        if name is None:
            name = os.path.splitext(template_filename)[0]
        if description is None:
            description = f"Template {name}"
        if tags is None:
            tags = []
        
        # Update metadata
        metadata = self._load_metadata()
        metadata["templates"][template_filename] = {
            "name": name,
            "description": description,
            "tags": tags,
            "created": datetime.datetime.now().isoformat()
        }
        self._save_metadata(metadata)
        
        return str(template_path)
    
    def list_templates(self, tag=None):
        """List all templates, optionally filtered by tag"""
        metadata = self._load_metadata()
        templates = []
        
        for filename, info in metadata["templates"].items():
            if tag is None or tag in info["tags"]:
                template_info = info.copy()
                template_info["filename"] = filename
                template_info["path"] = str(self.template_dir / filename)
                templates.append(template_info)
        
        return templates

# Initialize template manager
template_manager = TemplateManager(template_dir=TEMPLATES_PATH)
print("✅ TemplateManager initialized")

✅ TemplateManager initialized


## Helper Functions

Core utility functions for HTML generation and file handling:

In [5]:
def get_common_html_header(title="NoteMorphAI Template"):
    """Generate common HTML header with styles for all templates"""
    if TEMPLATE_DATA:
        # Use replace instead of format to avoid CSS brace conflicts
        header_template = TEMPLATE_DATA["common_styles"]["base_header"]
        return header_template.replace("{title}", title)
    else:
        # Fallback basic header
        return f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{title}</title>
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.0.0/css/all.min.css">
</head>"""

def open_in_browser(filepath):
    """Open HTML file in default browser"""
    abs_path = os.path.abspath(filepath)
    webbrowser.open(f"file://{abs_path}")
    print(f"✅ Template created: {abs_path}")

def preview_templates(directory="template_showcase"):
    """Preview all generated templates in browser"""
    template_dir = Path(directory)
    if not template_dir.exists():
        print(f"❌ Template directory '{directory}' not found")
        return
    
    html_files = list(template_dir.glob("*.html"))
    if not html_files:
        print(f"❌ No HTML templates found in '{directory}'")
        return
    
    print(f"🌐 Opening {len(html_files)} templates in browser...")
    for html_file in html_files:
        abs_path = html_file.absolute()
        webbrowser.open(f"file://{abs_path}")
        print(f"  ✅ Opened: {html_file.name}")

print("✅ Helper functions defined")

✅ Helper functions defined


## Climate Change Infographic Template

Creates visually appealing environmental education infographics:

In [6]:
def create_climate_change_infographic(output_path):
    """Create a climate change themed infographic using template data"""
    
    # Load template data or use fallback
    if TEMPLATE_DATA and "climate_change" in TEMPLATE_DATA["templates"]:
        climate_data = TEMPLATE_DATA["templates"]["climate_change"]
        title = climate_data["title"]
        subtitle = climate_data["subtitle"]
        content = climate_data["content"]
    else:
        # Fallback data
        title = "Climate Change Impact"
        subtitle = "Environmental & Social Consequences"
        content = {
            "definition": "Climate change refers to long-term shifts in temperatures and weather patterns.",
            "issues": [
                {"title": "🌡️ Rising Temperatures", "content": "Global average temperature has risen by 1.1°C"},
                {"title": "🌊 Sea Level Rise", "content": "Sea levels have risen 20cm since 1900"}
            ],
            "solutions": [
                "Transition to renewable energy sources",
                "Improve energy efficiency",
                "Protect natural ecosystems"
            ]
        }
    
    # Get template-specific styles
    climate_styles = ""
    if TEMPLATE_DATA and "climate_styles" in TEMPLATE_DATA["common_styles"]:
        climate_styles = TEMPLATE_DATA["common_styles"]["climate_styles"]
    
    html_content = get_common_html_header(title)
    
    # Add climate-specific styles
    if climate_styles:
        html_content = html_content.replace("</style>", f"{climate_styles}</style>")
    
    # Build the HTML content
    html_content += f"""
<body>
    <button class="print-button" onclick="window.print()">
        <i class="fas fa-print"></i>
    </button>

    <div class="notebook-page">
        <div class="washi-tape" style="top: 15px; left: 50px; width: 120px; background-color: #b3e5fc;"></div>
        <div class="washi-tape green" style="top: 10px; right: 80px; width: 100px; background-color: #b2dfdb;"></div>
        
        <div class="paper-clip">
            <i class="fas fa-paperclip"></i>
        </div>
        
        <div class="title-area">
            <h1 class="main-title">{title}</h1>
            <p class="subtitle">{subtitle}</p>
        </div>
        
        <div class="content-section">
            <div class="definition-box">
                <h2 class="section-title">What is Climate Change? <i class="fas fa-globe-americas"></i></h2>
                <p style="font-size: 18px; line-height: 1.7;">
                    {content["definition"]}
                </p>
            </div>
        </div>
        
        <div class="issues-grid">"""
    
    # Add issue cards
    for issue in content["issues"]:
        html_content += f"""
            <div class="issue-card">
                <div class="issue-header">
                    <span class="issue-icon">{issue["title"].split()[0]}</span>
                    {issue["title"][2:]}  
                </div>
                <div class="issue-content">
                    <p>{issue["content"]}</p>
                </div>
            </div>"""
    
    html_content += """
        </div>
        
        <div style="margin-top: 30px; padding: 20px; background: rgba(255, 193, 7, 0.2); border-radius: 15px;">
            <h2 style="color: #f57f17; margin-bottom: 15px;">Solutions & Actions 💡</h2>
            <ul style="font-size: 16px; line-height: 1.8;">"""
    
    # Add solutions
    for solution in content["solutions"]:
        html_content += f"<li>{solution}</li>"
    
    html_content += """
            </ul>
        </div>
    </div>
</body>
</html>"""
    
    # Write the HTML file
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(html_content)
    
    print(f"Climate Change infographic created at {output_path}")
    return output_path

print("✅ Climate Change template function defined")

✅ Climate Change template function defined


## Nutrition Step-by-Step Guide Template

Creates comprehensive nutrition guides with meal planning and health tips:

In [7]:
def create_nutrition_stepbystep(output_path):
    """Create a nutrition step-by-step guide using template data"""
    
    # Load template data or use fallback
    if TEMPLATE_DATA and "nutrition_guide" in TEMPLATE_DATA["templates"]:
        nutrition_data = TEMPLATE_DATA["templates"]["nutrition_guide"]
        title = nutrition_data["title"]
        subtitle = nutrition_data["subtitle"]
        steps = nutrition_data["steps"]
    else:
        # Fallback data
        title = "Nutrition Guide"
        subtitle = "Healthy Eating Made Simple"
        steps = [
            {"number": "1", "title": "Plan Your Meals", "content": "Start with meal planning.", "tips": ["Use a weekly planner"]},
            {"number": "2", "title": "Choose Whole Foods", "content": "Prioritize whole foods.", "tips": ["Fresh fruits and vegetables"]}
        ]
    
    # Get template-specific styles
    nutrition_styles = ""
    if TEMPLATE_DATA and "nutrition_styles" in TEMPLATE_DATA["common_styles"]:
        nutrition_styles = TEMPLATE_DATA["common_styles"]["nutrition_styles"]
    
    html_content = get_common_html_header(title)
    
    # Add nutrition-specific styles
    if nutrition_styles:
        html_content = html_content.replace("</style>", f"{nutrition_styles}</style>")
    
    # Build the HTML content
    html_content += f"""
<body>
    <button class="print-button" onclick="window.print()">
        <i class="fas fa-print"></i>
    </button>

    <div class="notebook-page">
        <div class="title-area">
            <h1 class="main-title">{title}</h1>
            <p class="subtitle">{subtitle}</p>
        </div>"""
    
    # Add step containers
    for step in steps:
        html_content += f"""
        <div class="step-container">
            <div class="step-number">{step["number"]}</div>
            <h3 class="step-title">{step["title"]}</h3>
            <p>{step["content"]}</p>
            <ul>"""
        
        for tip in step["tips"]:
            html_content += f"<li>{tip}</li>"
        
        html_content += "</ul></div>"
    
    html_content += """
    </div>
</body>
</html>"""
    
    # Write the HTML file
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(html_content)
    
    print(f"Nutrition guide created at {output_path}")
    return output_path

print("✅ Nutrition template function defined")

✅ Nutrition template function defined


## AI Cornell Notes Template

Creates structured Cornell-style notes optimized for technical education:

In [8]:
def create_ai_cornell(output_path):
    """Create AI Cornell notes using template data"""
    
    # Load template data or use fallback
    if TEMPLATE_DATA and "cornell_notes" in TEMPLATE_DATA["templates"]:
        cornell_data = TEMPLATE_DATA["templates"]["cornell_notes"]
        title = cornell_data["title"]
        subtitle = cornell_data["subtitle"]
        sections = cornell_data["sections"]
    else:
        # Fallback data
        title = "AI Cornell Notes"
        subtitle = "Structured Learning Template"
        sections = {
            "cue_column": ["Key Terms", "Questions"],
            "notes_content": ["Detailed notes go here"],
            "summary": "Summary section"
        }
    
    # Get template-specific styles
    cornell_styles = ""
    if TEMPLATE_DATA and "cornell_styles" in TEMPLATE_DATA["common_styles"]:
        cornell_styles = TEMPLATE_DATA["common_styles"]["cornell_styles"]
    
    html_content = get_common_html_header(title)
    
    # Add cornell-specific styles
    if cornell_styles:
        html_content = html_content.replace("</style>", f"{cornell_styles}</style>")
    
    # Build the HTML content
    html_content += f"""
<body>
    <button class="print-button" onclick="window.print()">
        <i class="fas fa-print"></i>
    </button>

    <div class="notebook-page">
        <div class="title-area">
            <h1 class="main-title">{title}</h1>
            <p class="subtitle">{subtitle}</p>
        </div>
        
        <div class="cornell-container">
            <div class="cue-column">
                <h3>Cue Column</h3>"""
    
    # Add cue items
    for item in sections["cue_column"]:
        html_content += f"<p>• {item}</p>"
    
    html_content += """
            </div>
            <div class="notes-column">
                <h3>Notes</h3>"""
    
    # Add notes content
    for note in sections["notes_content"]:
        html_content += f"<p>{note}</p>"
    
    html_content += f"""
            </div>
            <div class="summary-section">
                <h3>Summary</h3>
                <p>{sections["summary"]}</p>
            </div>
        </div>
    </div>
</body>
</html>"""
    
    # Write the HTML file
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(html_content)
    
    print(f"Cornell notes created at {output_path}")
    return output_path

print("✅ Cornell Notes template function defined")

✅ Cornell Notes template function defined


## Template Format Demonstrations

Individual demonstrations of each template format with clean output separation:

In [9]:
# Create output directory for template demonstrations
output_dir = Path("template_showcase")
output_dir.mkdir(exist_ok=True)

print("🚀 NoteMorphAI Template Format Showcase")
print("=" * 50)
print(f"Student: Komal Shahid")
print(f"Course: DSC670 - Machine Learning Applications")
print(f"Project: AI-Powered Note Transformation System")
print("=" * 50)
print(f"📁 Output directory: {output_dir.absolute()}")
print("\n✅ Setup complete - Ready to generate template formats")

🚀 NoteMorphAI Template Format Showcase
Student: Komal Shahid
Course: DSC670 - Machine Learning Applications
Project: AI-Powered Note Transformation System
📁 Output directory: /Users/komalshahid/Desktop/Bellevue University/DSC670/term_project/notely/milestones/milestone3/template_showcase

✅ Setup complete - Ready to generate template formats


### 📊 Infographic Template Format Demo

Demonstrates the **Visual/Creative** template format optimized for data presentation and educational content:

In [10]:
print("📊 Generating Infographic Template Format...")
print("Format: Visual/Creative - Infographic Layout")
print("Use Case: Data presentation, educational content, awareness campaigns")
print("Features: Visual elements, structured data sections, engaging design\n")

infographic_path = create_climate_change_infographic(str(output_dir / "infographic_format.html"))
template_manager.add_template(infographic_path, 
                             "Infographic Format", 
                             "Visual template for data presentation and education", 
                             ["infographic", "visual", "educational"])

print(f"✅ Infographic format created: {infographic_path}")
print("📋 Template registered in Template Manager")
print("🔗 Open the HTML file to view the visual layout\n")

📊 Generating Infographic Template Format...
Format: Visual/Creative - Infographic Layout
Use Case: Data presentation, educational content, awareness campaigns
Features: Visual elements, structured data sections, engaging design

Climate Change infographic created at template_showcase/infographic_format.html
✅ Infographic format created: template_showcase/infographic_format.html
📋 Template registered in Template Manager
🔗 Open the HTML file to view the visual layout



### 📚 Step-by-Step Guide Format Demo

Demonstrates the **Sequential Instruction** template format for tutorials and process documentation:

In [11]:
print("📚 Generating Step-by-Step Guide Format...")
print("Format: Professional - Sequential Instruction Layout")
print("Use Case: Tutorials, process documentation, how-to guides")
print("Features: Numbered steps, clear progression, actionable content\n")

stepguide_path = create_nutrition_stepbystep(str(output_dir / "stepguide_format.html"))
template_manager.add_template(stepguide_path, 
                             "Step-by-Step Guide Format", 
                             "Sequential instruction template for tutorials and processes", 
                             ["guide", "tutorial", "sequential"])

print(f"✅ Step-by-Step format created: {stepguide_path}")
print("📋 Template registered in Template Manager")
print("🔗 Open the HTML file to view the sequential layout\n")

📚 Generating Step-by-Step Guide Format...
Format: Professional - Sequential Instruction Layout
Use Case: Tutorials, process documentation, how-to guides
Features: Numbered steps, clear progression, actionable content

Nutrition guide created at template_showcase/stepguide_format.html
✅ Step-by-Step format created: template_showcase/stepguide_format.html
📋 Template registered in Template Manager
🔗 Open the HTML file to view the sequential layout



### 🎓 Cornell Notes Format Demo

Demonstrates the **Educational/Academic** template format for structured note-taking and study:

In [12]:
print("🎓 Generating Cornell Notes Format...")
print("Format: Educational - Structured Note-taking Layout")
print("Use Case: Academic notes, meeting notes, structured learning")
print("Features: Cue column, main notes area, summary section\n")

cornell_path = create_ai_cornell(str(output_dir / "cornell_format.html"))
template_manager.add_template(cornell_path, 
                             "Cornell Notes Format", 
                             "Structured educational template for note-taking and study", 
                             ["cornell", "academic", "notes"])

print(f"✅ Cornell Notes format created: {cornell_path}")
print("📋 Template registered in Template Manager")
print("🔗 Open the HTML file to view the structured layout\n")

🎓 Generating Cornell Notes Format...
Format: Educational - Structured Note-taking Layout
Use Case: Academic notes, meeting notes, structured learning
Features: Cue column, main notes area, summary section

Cornell notes created at template_showcase/cornell_format.html
✅ Cornell Notes format created: template_showcase/cornell_format.html
📋 Template registered in Template Manager
🔗 Open the HTML file to view the structured layout



### 🏛️ History Academic Notes Format Demo

Demonstrates the **Academic Timeline** template format for historical documentation:

In [13]:
# Import the history academic function from source
import sys
sys.path.append('../../src')
from showcase_templates import create_history_academic

print("🏛️ Generating History Academic Notes Format...")
print("Format: Academic - Historical Timeline Layout")
print("Use Case: History courses, research notes, chronological documentation")
print("Features: Timeline structure, date markers, contextual sections\n")

history_path = create_history_academic(str(output_dir / "history_academic.html"))
template_manager.add_template(history_path, 
                             "History Academic Format", 
                             "Structured academic template for historical timeline content", 
                             ["academic", "history", "timeline"])

print(f"✅ History Academic format created: history_academic.html")
print("📋 Template registered in Template Manager")
print("🔗 Open the HTML file to view the timeline layout\n")

🏛️ Generating History Academic Notes Format...
Format: Academic - Historical Timeline Layout
Use Case: History courses, research notes, chronological documentation
Features: Timeline structure, date markers, contextual sections

History academic notes created at template_showcase/history_academic.html
✅ History Academic format created: history_academic.html
📋 Template registered in Template Manager
🔗 Open the HTML file to view the timeline layout



### 👗 Fashion Industry Infographic Format Demo

Demonstrates **Social Impact Infographic** template format for awareness topics:

In [14]:
# Import the fashion infographic function from source
from showcase_templates import create_fashion_infographic

print("👗 Generating Fashion Industry Infographic Format...")
print("Format: Infographic - Social Impact Layout")
print("Use Case: Social awareness, environmental topics, impact documentation")
print("Features: Visual impact cards, statistical displays, awareness sections\n")

fashion_path = create_fashion_infographic(str(output_dir / "fashion_infographic.html"))
template_manager.add_template(fashion_path, 
                             "Fashion Infographic Format", 
                             "Visual template for social and environmental impact topics", 
                             ["infographic", "social-impact", "environment"])

print(f"✅ Fashion Infographic format created: fashion_infographic.html")
print("📋 Template registered in Template Manager")
print("🔗 Open the HTML file to view the impact layout\n")

👗 Generating Fashion Industry Infographic Format...
Format: Infographic - Social Impact Layout
Use Case: Social awareness, environmental topics, impact documentation
Features: Visual impact cards, statistical displays, awareness sections

Fast Fashion infographic created at template_showcase/fashion_infographic.html
✅ Fashion Infographic format created: fashion_infographic.html
📋 Template registered in Template Manager
🔗 Open the HTML file to view the impact layout



### 📋 Project Planning Notes Format Demo

Demonstrates **Project Management** template format for planning documentation:

In [15]:
# Import the project planning function from source
from showcase_templates import create_project_notes

print("📋 Generating Project Planning Notes Format...")
print("Format: Management - Project Planning Layout")
print("Use Case: Project management, team planning, milestone tracking")
print("Features: Task sections, timeline views, progress tracking\n")

project_path = create_project_notes(str(output_dir / "project_planning.html"))
template_manager.add_template(project_path, 
                             "Project Planning Format", 
                             "Structured template for project management and planning", 
                             ["project", "planning", "management"])

print(f"✅ Project Planning format created: project_planning.html")
print("📋 Template registered in Template Manager")
print("🔗 Open the HTML file to view the planning layout\n")

📋 Generating Project Planning Notes Format...
Format: Management - Project Planning Layout
Use Case: Project management, team planning, milestone tracking
Features: Task sections, timeline views, progress tracking

Project notes template created at template_showcase/project_planning.html
✅ Project Planning format created: project_planning.html
📋 Template registered in Template Manager
🔗 Open the HTML file to view the planning layout



### 📊 Template Format Summary

Overview of all generated template formats and Template Manager status:

### 🌐 Template Preview

Open all generated templates in your browser for easy viewing:

In [16]:
# Preview all generated templates in browser
print("🌐 Template Preview Tool")
print("=" * 25)

# Check if templates exist
template_dir = Path("template_showcase")
if template_dir.exists():
    html_files = list(template_dir.glob("*.html"))
    print(f"📁 Found {len(html_files)} templates to preview")
    
    # List available templates
    for i, html_file in enumerate(html_files, 1):
        file_size = html_file.stat().st_size
        print(f"  {i}. {html_file.name} ({file_size:,} bytes)")
    
    print("\n🚀 Opening templates in browser...")
    preview_templates()
    
    print("\n💡 Tips:")
    print("  • Templates open in separate browser tabs")
    print("  • Use browser print function to save as PDF")
    print("  • Templates are fully responsive and interactive")
else:
    print("❌ No templates found. Run the demo cells above first!")
    print("💡 Execute the template generation cells to create templates.")

🌐 Template Preview Tool
📁 Found 6 templates to preview
  1. fashion_infographic.html (13,368 bytes)
  2. cornell_format.html (6,022 bytes)
  3. project_planning.html (15,479 bytes)
  4. history_academic.html (16,883 bytes)
  5. infographic_format.html (8,267 bytes)
  6. stepguide_format.html (6,928 bytes)

🚀 Opening templates in browser...
🌐 Opening 6 templates in browser...
  ✅ Opened: fashion_infographic.html
  ✅ Opened: cornell_format.html
  ✅ Opened: project_planning.html
  ✅ Opened: history_academic.html
  ✅ Opened: infographic_format.html
  ✅ Opened: stepguide_format.html

💡 Tips:
  • Templates open in separate browser tabs
  • Use browser print function to save as PDF
  • Templates are fully responsive and interactive


In [17]:
print("📊 Template Format Generation Summary")
print("=" * 45)

# Display Template Manager inventory
print("\n📋 Template Manager Inventory:")
templates = template_manager.list_templates()
for i, template in enumerate(templates, 1):
    print(f"  {i}. {template['name']}")
    print(f"     📝 {template['description']}")
    print(f"     🏷️  Tags: {', '.join(template['tags'])}")
    print()

print("\n📈 Project Statistics:")
print(f"🎯 Total template formats: {len(templates)}")
print(f"📁 Storage location: {template_manager.template_dir.absolute()}")
print(f"📁 Demo output: {output_dir.absolute()}")

# Template data status
if TEMPLATE_DATA:
    print(f"\n📄 Template definitions loaded: {len(TEMPLATE_DATA['templates'])}")
    print(f"🎨 Style variants available: {len(TEMPLATE_DATA['common_styles'])}")
else:
    print("\n⚠️ Template data not loaded - used fallback content")

print("\n🎉 NoteMorphAI template format showcase complete!")
print("💡 These formats can be adapted for any content type")
print("🔗 Open the HTML files in your browser to explore each format")

📊 Template Format Generation Summary

📋 Template Manager Inventory:
  1. Infographic Format
     📝 Visual template for data presentation and education
     🏷️  Tags: infographic, visual, educational

  2. Step-by-Step Guide Format
     📝 Sequential instruction template for tutorials and processes
     🏷️  Tags: guide, tutorial, sequential

  3. Cornell Notes Format
     📝 Structured educational template for note-taking and study
     🏷️  Tags: cornell, academic, notes

  4. History Academic Format
     📝 Structured academic template for historical timeline content
     🏷️  Tags: academic, history, timeline

  5. Fashion Infographic Format
     📝 Visual template for social and environmental impact topics
     🏷️  Tags: infographic, social-impact, environment

  6. Project Planning Format
     📝 Structured template for project management and planning
     🏷️  Tags: project, planning, management


📈 Project Statistics:
🎯 Total template formats: 6
📁 Storage location: /Users/komalshahid/Des